# Global Health Progress, 2000–2019: A Data Story from the World Bank's Health, Nutrition and Population Statistics

**3520872** | CSCU9M3 — Scripting for Data Science | Autumn 2026

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Colab always has a module called google.colab loaded automatically, so I
# can use that to check if the notebook is running in Colab or not, and
# point at the right folder for the data either way.
running_in_colab = "google.colab" in sys.modules

if running_in_colab:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Untitled folder/data/hnp_stats"
else:
    DATA_DIR = "data/hnp_stats"

## Executive Summary

_[Written last, after the findings below are known — max 200 words.]_

## Introduction

_[Written last, after the findings below are known — max 200 words.]_

## Data Cleaning

The World Bank's Health, Nutrition and Population Statistics (HNP) data comes as several linked CSV files: a main data file (one row per country/indicator, one column per year 1960–2021), a country metadata file, and an indicator metadata file. The main file needs cleaning before any analysis. It mixes real countries with regional and income-group aggregates in the same rows, it contains hundreds of indicators and six decades of years that this report does not use, and many country/indicator combinations have no data at all for the years we care about.

In [ ]:
data = pd.read_csv(f"{DATA_DIR}/HNP_StatsData.csv")
country_info = pd.read_csv(f"{DATA_DIR}/HNP_StatsCountry.csv")

# Both files end every line with a trailing comma, so pandas reads an extra
# empty column called "Unnamed: xx" at the end. I get rid of those columns
# before doing anything else with the data.
extra_columns = [col for col in data.columns if col.startswith("Unnamed")]
data = data.drop(columns=extra_columns)

extra_columns = [col for col in country_info.columns if col.startswith("Unnamed")]
country_info = country_info.drop(columns=extra_columns)

print("Main data file:", data.shape)
print("Country info file:", country_info.shape)

Removing non-country aggregates. Rows such as "Africa Eastern and Southern" or "High income" are regional/income-group aggregates, not countries, and would distort any country-level comparison (e.g. a regional aggregate being counted alongside the countries that make it up). An alternative would be to hand-pick a list of aggregate names to drop, but this was rejected because it is brittle and error-prone, it relies on recognising every aggregate by hand and would silently miss any that are mis-spelled or renamed. The country metadata file already distinguishes real countries from aggregates, every real country has a Region value, while every aggregate has a blank one. Filtering on that column is a rule based on the data rather than a manual list.

In [ ]:
# Rows with no Region are not real countries, they are totals for a whole
# region or income group (e.g. "Africa Eastern and Southern"), so I filter
# those out here and keep just the columns I need from the country info.
real_countries = country_info[country_info["Region"].notna()]
real_countries = real_countries[["Country Code", "Region", "Income Group"]]

data = data.merge(real_countries, on="Country Code", how="inner")

print("Rows after removing region/income totals:", len(data))
print("Number of countries left:", data["Country Code"].nunique())

Choosing indicators. The full dataset covers hundreds of indicators spanning population, nutrition, disease, and health systems, far more than a 1500-word report can cover. Five indicators were chosen to address one question (has global health improved since 2000, and is that improvement shared equally?), life expectancy at birth, under-5 mortality, measles immunisation coverage, fertility rate, and current health expenditure (% of GDP). These were chosen for good data coverage (checked below), and because together they cover outcomes (mortality, life expectancy), a driver (health spending), and context (fertility, immunisation).

In [ ]:
indicators_wanted = {
    "SP.DYN.LE00.IN": "Life expectancy at birth (years)",
    "SH.DYN.MORT": "Under-5 mortality (per 1,000 live births)",
    "SH.IMM.MEAS": "Measles immunisation (% of children 12-23 months)",
    "SP.DYN.TFRT.IN": "Fertility rate (births per woman)",
    "SH.XPD.CHEX.GD.ZS": "Current health expenditure (% of GDP)",
}

data = data[data["Indicator Code"].isin(indicators_wanted)]
data["Indicator Name"] = data["Indicator Code"].map(indicators_wanted)
print("Rows after keeping only the 5 indicators I want:", len(data))

Choosing the year range. Current health expenditure (% of GDP) is only reported from 2000 onwards (an earlier, broader methodology didn't track it), so 2000 is the start of the comparison across all five indicators, using an earlier start would mean comparing on four indicators for most of the window and five only at the end. 2019 is used as the end point rather than the most recent years (2020-2021) because COVID-19 caused sharp, temporary disruption to both health outcomes and data collection in those years, including them would make a "has health improved" comparison partly about the pandemic rather than the longer-run trend this report is about.

In [ ]:
YEARS = [str(year) for year in range(2000, 2020)]
columns_to_keep = ["Country Name", "Country Code", "Region", "Income Group",
                    "Indicator Name", "Indicator Code"] + YEARS
data = data[columns_to_keep]
print("Columns kept:", list(data.columns))

Handling missing values. For the before/after comparisons used later in this report, a country/indicator row is only usable if it has a value in both endpoint years, 2000 and 2019. Rows missing either endpoint are dropped rather than filled in by interpolation, interpolating would invent a trend for a country that never reported that indicator reliably, and that gap in reporting is a real feature of the data rather than something to paper over.

In [ ]:
missing_2000 = data["2000"].isna().sum()
missing_2019 = data["2019"].isna().sum()
print("Rows missing a 2000 value:", missing_2000)
print("Rows missing a 2019 value:", missing_2019)

clean_data = data.dropna(subset=["2000", "2019"])
clean_data = clean_data.reset_index(drop=True)

percent_kept = round(len(clean_data) / len(data) * 100)
print("Rows kept:", len(clean_data), "out of", len(data), f"({percent_kept}%)")

In [ ]:
clean_data[["Country Name", "Region", "Income Group", "Indicator Name", "2000", "2019"]].head()

clean_data is the dataset used for the rest of this report: real countries only, five indicators, years 2000–2019, with both endpoint years guaranteed present for every row. Intermediate years (2001–2018) may still contain gaps for some countries, this is fine because only the Exploratory Data Analysis section uses them, as year-by-year averages rather than individual country values.

## Exploratory Data Analysis

This section looks at what clean_data actually contains before the descriptive analytics questions. Three things are worth checking first, how much of each indicator is missing, how the 217 countries split across regions, and what the spread of a typical indicator looks like across countries.

In [ ]:
# I check missingness on data (the 1085 rows before I dropped the ones
# missing an endpoint year), grouped by indicator, so I can see if the
# missing values are spread evenly or concentrated in one indicator.
rows_missing_an_endpoint = data[data["2000"].isna() | data["2019"].isna()]
missing_by_indicator = rows_missing_an_endpoint.groupby("Indicator Name").size()
total_by_indicator = data.groupby("Indicator Name").size()

missing_percent = (missing_by_indicator / total_by_indicator * 100).round(1)
print(missing_percent)

plt.bar(missing_percent.index, missing_percent.values)
plt.ylabel("Percent of countries missing a 2000 or 2019 value")
plt.xticks(rotation=45, ha="right")
plt.show()

Checking missing data by indicator. Current health expenditure is missing for 37 of 217 countries, close to double the missing rate of life expectancy and fertility rate, which are each missing only 18. This is because health expenditure is a newer indicator with less historical reporting behind it, so any result that uses it rests on a smaller set of countries than the other four indicators.

In [ ]:
countries_per_region = clean_data.groupby("Region")["Country Code"].nunique()
countries_per_region = countries_per_region.sort_values()
print(countries_per_region)

plt.bar(countries_per_region.index, countries_per_region.values)
plt.ylabel("Number of countries")
plt.xticks(rotation=45, ha="right")
plt.show()

Checking country counts by region. The regions are far from equal in size, North America has 3 countries and South Asia has 8, while Europe & Central Asia has 56. This means a difference between two small regions could come from just one or two countries moving, so regional comparisons later in this report lean more on the larger regions.

In [ ]:
life_expectancy_2019 = clean_data[clean_data["Indicator Name"] == "Life expectancy at birth (years)"]["2019"]
fertility_2019 = clean_data[clean_data["Indicator Name"] == "Fertility rate (births per woman)"]["2019"]

print("Life expectancy, mean:", round(life_expectancy_2019.mean(), 1), "median:", round(life_expectancy_2019.median(), 1))
print("Fertility rate, mean:", round(fertility_2019.mean(), 2), "median:", round(fertility_2019.median(), 2))

plt.hist(life_expectancy_2019, bins=15)
plt.xlabel("Life expectancy at birth in 2019 (years)")
plt.ylabel("Number of countries")
plt.show()

plt.hist(fertility_2019, bins=15)
plt.xlabel("Fertility rate in 2019 (births per woman)")
plt.ylabel("Number of countries")
plt.show()

Checking the shape of two indicators. Life expectancy in 2019 has a mean of 73.0 years against a median of 74.2, so the mean sits below the median, a smaller group of countries with much lower life expectancy pulls it down. Fertility rate shows the opposite pattern, a mean of 2.62 births per woman against a median of 2.19, so a smaller group of high fertility countries pulls the mean up instead. This means the median is the more representative figure for a typical country on both indicators, and the descriptive analytics section below uses that where it matters.

## Descriptive Analytics

_[Phase 3 — to be completed next.]_

## Suggestions

_[Phase 4 — to be completed next.]_

## Bibliography

_[Phase 6 — to be completed last.]_